# Notebook 02: Preprocesamiento de Datos

**Objetivo**: Limpiar y preparar el dataset para entrenamiento de modelos.

**Contenido**:
1. Carga del dataset desde CSV
2. Filtrado de clases válidas (5 emociones principales)
3. Eliminación de columnas no predictivas (relacionales + metadata)
4. Manejo de valores nulos (dropear si <10 filas)
5. Conversión de columnas a numérico (Length, Loudness, Time signature, Key, Explicit)
6. Encoding de columnas categóricas (Genre)
7. Limpieza de texto (lyrics)
8. Split estratificado (80/20)
9. Exportación a parquet

---

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Carga del Dataset Original

In [2]:
# Cargar dataset completo
df = pd.read_csv('../data/spotify_dataset.csv')

print(f"✓ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Columnas: {len(df.columns)} columnas totales")

✓ Dataset cargado: 551,443 filas × 39 columnas
Columnas: 39 columnas totales


## 3. Filtrado de Clases Válidas

**Decisión técnica**: Mantener solo las 5 emociones principales (joy, sadness, anger, fear, love) con >5K ejemplos cada una.

**Justificación** (del notebook 01):
- Clases minoritarias (<5K ejemplos) son insuficientes para stratified split 80/20
- Ratio de desbalanceo mejora de >200,000x a ~7.5x (manejable con class_weight='balanced')

In [3]:
# Clases válidas (>5K ejemplos según EDA)
valid_emotions = ['joy', 'sadness', 'anger', 'fear', 'love']

print("=" * 60)
print("FILTRADO DE CLASES")
print("=" * 60)
print(f"Clases a mantener: {valid_emotions}")
print()

# Contar filas antes del filtrado
rows_before = len(df)

# Filtrar
df_filtered = df[df['emotion'].isin(valid_emotions)].copy()

# Contar filas después del filtrado
rows_after = len(df_filtered)
rows_removed = rows_before - rows_after
pct_removed = (rows_removed / rows_before) * 100

print(f"Filas antes del filtrado: {rows_before:,}")
print(f"Filas después del filtrado: {rows_after:,}")
print(f"Filas eliminadas: {rows_removed:,} ({pct_removed:.2f}%)")
print()

print(f"✓ Dataset filtrado: {len(df_filtered):,} filas")

FILTRADO DE CLASES
Clases a mantener: ['joy', 'sadness', 'anger', 'fear', 'love']

Filas antes del filtrado: 551,443
Filas después del filtrado: 545,825
Filas eliminadas: 5,618 (1.02%)

✓ Dataset filtrado: 545,825 filas


## 4. Eliminación de Columnas No Predictivas

**Decisión técnica**: Eliminar columnas relacionales y de metadata que no aportan información sobre el sentiment.

### Columnas a eliminar:
1. **Relacionales** (9 cols): Similar Artist 1/2/3, Similar Song 1/2/3, Similarity Score 1/2/3
2. **Metadata** (4 cols): Artist(s), song, Album, Release Date

**Justificación**:
- Relacionales: Referencian otras filas, no son features intrínsecas, riesgo de data leakage
- Metadata: Identificadores y fechas NO afectan el sentiment intrínseco de la letra

In [4]:
# Definir columnas a eliminar
relational_cols = [
    'Similar Artist 1', 'Similar Song 1', 'Similarity Score 1',
    'Similar Artist 2', 'Similar Song 2', 'Similarity Score 2',
    'Similar Artist 3', 'Similar Song 3', 'Similarity Score 3'
]

metadata_cols = ['Artist(s)', 'song', 'Album', 'Release Date']

cols_to_drop = relational_cols + metadata_cols

print("=" * 60)
print("ELIMINACIÓN DE COLUMNAS NO PREDICTIVAS")
print("=" * 60)
print(f"Total de columnas a eliminar: {len(cols_to_drop)}")
print()
print(f"  - Relacionales: {len(relational_cols)} columnas")
print(f"  - Metadata: {len(metadata_cols)} columnas")
print()

# Verificar cuáles existen
existing_cols = [col for col in cols_to_drop if col in df_filtered.columns]
print(f"Columnas existentes a eliminar: {len(existing_cols)}")
for col in existing_cols:
    print(f"   - {col}")

# Eliminar
df_filtered = df_filtered.drop(columns=existing_cols)

print()
print(f"✓ Columnas eliminadas. Columnas restantes: {len(df_filtered.columns)}")

ELIMINACIÓN DE COLUMNAS NO PREDICTIVAS
Total de columnas a eliminar: 13

  - Relacionales: 9 columnas
  - Metadata: 4 columnas

Columnas existentes a eliminar: 13
   - Similar Artist 1
   - Similar Song 1
   - Similarity Score 1
   - Similar Artist 2
   - Similar Song 2
   - Similarity Score 2
   - Similar Artist 3
   - Similar Song 3
   - Similarity Score 3
   - Artist(s)
   - song
   - Album
   - Release Date

✓ Columnas eliminadas. Columnas restantes: 26


## 5. Manejo de Valores Nulos

**Estrategia**: Si hay <0.1% filas con nulos, eliminarlas.

In [5]:
# Verificar nulos
# Umbral: si las filas con nulos representan menos del 0.1% del dataset,
# es más limpio eliminarlas que imputar con tan pocos datos.
NULL_DROP_THRESHOLD = 0.001  # 0.1%

null_counts = df_filtered.isnull().sum()
total_nulls = null_counts.sum()
rows_with_nulls = df_filtered.isnull().any(axis=1).sum()
null_pct = rows_with_nulls / len(df_filtered)

print("=" * 60)
print("MANEJO DE VALORES NULOS")
print("=" * 60)
print(f"Total de valores nulos: {total_nulls:,}")
print(f"Filas con al menos un nulo: {rows_with_nulls:,} ({null_pct*100:.4f}%)")
print(f"Umbral para eliminación: {NULL_DROP_THRESHOLD*100:.1f}% ({int(len(df_filtered)*NULL_DROP_THRESHOLD):,} filas)")
print()

if rows_with_nulls > 0:
    if null_pct < NULL_DROP_THRESHOLD:
        print(f"\U0001f5d1\ufe0f  Decisión: ELIMINAR las {rows_with_nulls} filas con nulos ({null_pct*100:.4f}% < {NULL_DROP_THRESHOLD*100:.1f}%)")
        print("   Imputar con tan pocos datos introduce ruido innecesario.")
        df_filtered = df_filtered.dropna()
        print(f"\u2713 Filas eliminadas. Dataset limpio: {len(df_filtered):,} filas")
    else:
        print(f"\u26a0\ufe0f  Hay {rows_with_nulls} filas con nulos ({null_pct*100:.4f}% \u2265 {NULL_DROP_THRESHOLD*100:.1f}%). Aplicando imputaci\u00f3n:")
        print()

        for col in null_counts[null_counts > 0].index:
            if col == 'text':
                df_filtered[col] = df_filtered[col].fillna('')
                print(f"   - {col}: nulos \u2192 string vac\u00edo")
            elif df_filtered[col].dtype in ['int64', 'float64']:
                median_val = df_filtered[col].median()
                df_filtered[col] = df_filtered[col].fillna(median_val)
                print(f"   - {col}: nulos \u2192 mediana ({median_val:.2f})")
            else:
                df_filtered[col] = df_filtered[col].fillna('unknown')
                print(f"   - {col}: nulos \u2192 'unknown'")

        print()
        print(f"\u2713 Imputaci\u00f3n completada. Nulos restantes: {df_filtered.isnull().sum().sum()}")
else:
    print("\u2705 No se encontraron valores nulos.")

print()
print(f"Dataset actual: {len(df_filtered):,} filas \u00d7 {len(df_filtered.columns)} columnas")


MANEJO DE VALORES NULOS
Total de valores nulos: 8
Filas con al menos un nulo: 8 (0.0015%)
Umbral para eliminación: 0.1% (545 filas)

🗑️  Decisión: ELIMINAR las 8 filas con nulos (0.0015% < 0.1%)
   Imputar con tan pocos datos introduce ruido innecesario.
✓ Filas eliminadas. Dataset limpio: 545,817 filas

Dataset actual: 545,817 filas × 26 columnas


## 6. Conversión de Columnas a Numérico

**Objetivo**: Maximizar features numéricas para mejorar poder predictivo.

### Conversiones a realizar:
1. **Length**: "03:47" → 227 segundos
2. **Loudness (db)**: "-6.85db" → -6.85 (float)
3. **Time signature**: "4/4" → 4 (numerador)
4. **Key**: "D min" → label encoding (0-23: 12 tonos × 2 modos)
5. **Explicit**: "Yes"/"No" → 1/0

In [6]:
def convert_length_to_seconds(length_str):
    """
    Convierte duración en formato MM:SS a segundos totales.
    Ejemplo: "03:47" → 227 segundos
    """
    if pd.isna(length_str) or length_str == '':
        return np.nan
    
    try:
        parts = str(length_str).strip().split(':')
        if len(parts) == 2:
            minutes, seconds = int(parts[0]), int(parts[1])
            return minutes * 60 + seconds
        elif len(parts) == 1:
            # Solo segundos
            return int(parts[0])
        else:
            return np.nan
    except:
        return np.nan

print("=" * 60)
print("CONVERSIÓN A NUMÉRICO: Length")
print("=" * 60)

sample_before = df_filtered['Length'].iloc[0]
df_filtered['Length_seconds'] = df_filtered['Length'].apply(convert_length_to_seconds)
sample_after = df_filtered['Length_seconds'].iloc[0]

print(f"Ejemplo: '{sample_before}' → {sample_after} segundos")
print(f"✓ Columna 'Length_seconds' creada (numérica)")

# Drop columna original
df_filtered = df_filtered.drop(columns=['Length'])
print(f"✓ Columna original 'Length' eliminada")


CONVERSIÓN A NUMÉRICO: Length
Ejemplo: '03:47' → 227 segundos
✓ Columna 'Length_seconds' creada (numérica)
✓ Columna original 'Length' eliminada


In [7]:
def convert_loudness_to_float(loudness_str):
    """
    Extrae valor numérico de loudness.
    Ejemplo: "-6.85db" → -6.85
    """
    if pd.isna(loudness_str):
        return np.nan
    
    try:
        # Remover "db" y otros caracteres no numéricos excepto -, .
        clean_str = re.sub(r'[^0-9.\-]', '', str(loudness_str))
        return float(clean_str)
    except:
        return np.nan

print("=" * 60)
print("CONVERSIÓN A NUMÉRICO: Loudness (db)")
print("=" * 60)

sample_before = df_filtered['Loudness (db)'].iloc[0]
df_filtered['Loudness'] = df_filtered['Loudness (db)'].apply(convert_loudness_to_float)
sample_after = df_filtered['Loudness'].iloc[0]

print(f"Ejemplo: '{sample_before}' → {sample_after}")
print(f"✓ Columna 'Loudness' creada (numérica)")

# Drop columna original
df_filtered = df_filtered.drop(columns=['Loudness (db)'])
print(f"✓ Columna original 'Loudness (db)' eliminada")


CONVERSIÓN A NUMÉRICO: Loudness (db)
Ejemplo: '-6.85db' → -6.85
✓ Columna 'Loudness' creada (numérica)
✓ Columna original 'Loudness (db)' eliminada


In [8]:
def convert_time_signature(signature_str):
    """
    Extrae numerador del compás.
    Ejemplo: "4/4" → 4
    """
    if pd.isna(signature_str):
        return np.nan
    
    try:
        # Extraer numerador (antes del /)
        numerator = str(signature_str).split('/')[0].strip()
        return int(numerator)
    except:
        return np.nan

print("=" * 60)
print("CONVERSIÓN A NUMÉRICO: Time signature")
print("=" * 60)

sample_before = df_filtered['Time signature'].iloc[0]
df_filtered['Time_signature'] = df_filtered['Time signature'].apply(convert_time_signature)
sample_after = df_filtered['Time_signature'].iloc[0]

print(f"Ejemplo: '{sample_before}' → {sample_after}")
print(f"✓ Columna 'Time_signature' creada (numérica)")

# Drop columna original
df_filtered = df_filtered.drop(columns=['Time signature'])
print(f"✓ Columna original 'Time signature' eliminada")


CONVERSIÓN A NUMÉRICO: Time signature
Ejemplo: '4/4' → 4
✓ Columna 'Time_signature' creada (numérica)
✓ Columna original 'Time signature' eliminada


In [9]:
def convert_key_to_tonic_mode(key_str):
    """
    Convierte tonalidad musical a dos features independientes:
    - Key_tonic: tónica (0-11, cromático desde C)
    - Key_mode : modo (0 = mayor, 1 = menor)

    Separar en dos columnas es correcto porque tónica y modo son dimensiones
    musicalmente independientes. Un encoding ordinal único (0-23) crearía
    distancias erróneas: C_maj (0) y B_min (23) parecerían muy lejanos cuando
    musicalmente son adyacentes.

    Ejemplo: "D min" → tonic=2, mode=1
    """
    if pd.isna(key_str) or key_str == '':
        return np.nan, np.nan

    key_str = str(key_str).strip().lower()

    tone_map = {
        'c': 0, 'c#': 1, 'db': 1,
        'd': 2, 'd#': 3, 'eb': 3,
        'e': 4,
        'f': 5, 'f#': 6, 'gb': 6,
        'g': 7, 'g#': 8, 'ab': 8,
        'a': 9, 'a#': 10, 'bb': 10,
        'b': 11
    }

    is_minor = 'min' in key_str
    tone_str = key_str.replace('major', '').replace('minor', '').replace('min', '').replace('maj', '').strip()
    tone_value = tone_map.get(tone_str, np.nan)

    if pd.isna(tone_value):
        return np.nan, np.nan

    return int(tone_value), int(is_minor)


print("=" * 60)
print("CONVERSIÓN A NUMÉRICO: Key → Key_tonic + Key_mode")
print("=" * 60)

sample_before = df_filtered['Key'].iloc[0]
tonic_series, mode_series = zip(*df_filtered['Key'].apply(convert_key_to_tonic_mode))
df_filtered['Key_tonic'] = tonic_series   # 0-11: C, C#, D, ... B
df_filtered['Key_mode']  = mode_series    # 0=mayor, 1=menor

sample_tonic = df_filtered['Key_tonic'].iloc[0]
sample_mode  = df_filtered['Key_mode'].iloc[0]

print(f"Ejemplo: '{sample_before}' → tonic={sample_tonic}, mode={sample_mode}")
print(f"✓ Columna 'Key_tonic' creada (0-11, tónica cromática)")
print(f"✓ Columna 'Key_mode'  creada (0=mayor, 1=menor)")

# Drop columnas originales
df_filtered = df_filtered.drop(columns=['Key'])
if 'Key_encoded' in df_filtered.columns:
    df_filtered = df_filtered.drop(columns=['Key_encoded'])
print(f"✓ Columnas originales 'Key' y 'Key_encoded' eliminadas")


CONVERSIÓN A NUMÉRICO: Key → Key_tonic + Key_mode
Ejemplo: 'D min' → tonic=2, mode=1
✓ Columna 'Key_tonic' creada (0-11, tónica cromática)
✓ Columna 'Key_mode'  creada (0=mayor, 1=menor)
✓ Columnas originales 'Key' y 'Key_encoded' eliminadas


In [10]:
print("=" * 60)
print("CONVERSIÓN A NUMÉRICO: Explicit")
print("=" * 60)

sample_before = df_filtered['Explicit'].iloc[0]
df_filtered['Explicit_binary'] = df_filtered['Explicit'].apply(
    lambda x: 1 if str(x).strip().lower() in ['yes', 'true', '1'] else 0
)
sample_after = df_filtered['Explicit_binary'].iloc[0]

print(f"Ejemplo: '{sample_before}' → {sample_after}")
print(f"✓ Columna 'Explicit_binary' creada (0/1)")

# Drop columna original
df_filtered = df_filtered.drop(columns=['Explicit'])
print(f"✓ Columna original 'Explicit' eliminada")

print()
print(f"✅ Conversiones completadas. Columnas actuales: {len(df_filtered.columns)}")

CONVERSIÓN A NUMÉRICO: Explicit
Ejemplo: 'No' → 0
✓ Columna 'Explicit_binary' creada (0/1)
✓ Columna original 'Explicit' eliminada

✅ Conversiones completadas. Columnas actuales: 27


## 7. Encoding de Columnas Categóricas

**Columna**: Genre (si tiene <20 valores únicos → one-hot, sino → frequency encoding)

In [11]:
print("=" * 60)
print("ENCODING DE COLUMNA CATEGÓRICA: Genre")
print("=" * 60)

n_unique_genres = df_filtered['Genre'].nunique()
print(f"Géneros únicos: {n_unique_genres}")
print()

if n_unique_genres < 20:
    print(f"📊 Aplicando ONE-HOT encoding (<20 géneros)")
    genre_dummies = pd.get_dummies(df_filtered['Genre'], prefix='Genre')
    df_filtered = pd.concat([df_filtered, genre_dummies], axis=1)
    df_filtered = df_filtered.drop(columns=['Genre'])
    print(f"✓ {len(genre_dummies.columns)} columnas one-hot creadas")
    GENRE_ENCODING = 'onehot'
else:
    print(f"📊 Aplicando FREQUENCY encoding (>20 géneros)")
    print()
    print("  ⚠️  Nota: el mapeo de frecuencias se calcula DESPUÉS del split")
    print("     (solo desde train_df) para evitar data leakage.")
    print("     La columna Genre se conserva hasta el split.")
    # NO mapear aquí — se hará post-split para no contaminar test
    GENRE_ENCODING = 'frequency'

print()
print(f"Dataset actual: {len(df_filtered):,} filas × {len(df_filtered.columns)} columnas")


ENCODING DE COLUMNA CATEGÓRICA: Genre


Géneros únicos: 3092

📊 Aplicando FREQUENCY encoding (>20 géneros)

  ⚠️  Nota: el mapeo de frecuencias se calcula DESPUÉS del split
     (solo desde train_df) para evitar data leakage.
     La columna Genre se conserva hasta el split.

Dataset actual: 545,817 filas × 27 columnas


## 8. Limpieza de Texto (Lyrics)

Preprocesamiento básico de la columna `text` (lyrics) para embeddings.

In [12]:
def clean_lyrics(text):
    """
    Limpia el texto de lyrics para sentence-transformers:
    - NO convierte a minúsculas: all-MiniLM-L6-v2 fue entrenado con texto
      en mixed case y su tokenizador WordPiece usa vocabulario case-sensitive.
      Bajar a minúsculas degrada la representación.
    - Remueve URLs.
    - Remueve solo caracteres de control y caracteres no imprimibles;
      preserva letras Unicode (acentos, ñ, ü, etc.) para no dañar canciones
      en español, francés, portugués, etc.
    - Colapsa espacios múltiples.
    """
    if not isinstance(text, str) or text == '':
        return ''

    # Remover URLs
    import re
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remover caracteres de control y no imprimibles (pero preservar Unicode)
    text = re.sub(r'[\x00-\x1f\x7f]', ' ', text)

    # Remover espacios múltiples
    text = re.sub(r'\s+', ' ', text)

    # Trim
    text = text.strip()

    return text

print("=" * 60)
print("LIMPIEZA DE TEXTO (LYRICS)")
print("=" * 60)

# Verificar si existe columna 'text'
if 'text' in df_filtered.columns:
    # Aplicar limpieza
    df_filtered['text_clean'] = df_filtered['text'].apply(clean_lyrics)
    
    # Mostrar ejemplo de antes/después
    print("Ejemplo de limpieza (primera fila no vacía):")
    non_empty = df_filtered[df_filtered['text'].str.len() > 50]
    if len(non_empty) > 0:
        sample_idx = non_empty.index[0]
        print()
        print(f"ANTES (primeros 150 chars):")
        print(f"{df_filtered.loc[sample_idx, 'text'][:150]}...")
        print()
        print(f"DESPUÉS (primeros 150 chars):")
        print(f"{df_filtered.loc[sample_idx, 'text_clean'][:150]}...")
    
    print()
    print(f"✓ Columna 'text_clean' creada ({len(df_filtered):,} filas procesadas)")
else:
    print("⚠️  Columna 'text' no encontrada. Omitiendo limpieza de lyrics.")

LIMPIEZA DE TEXTO (LYRICS)
Ejemplo de limpieza (primera fila no vacía):

ANTES (primeros 150 chars):
Friends told her she was better off at the bottom of a river Than in a bed with him He said "Until you try both, you won't know what you like better W...

DESPUÉS (primeros 150 chars):
Friends told her she was better off at the bottom of a river Than in a bed with him He said "Until you try both, you won't know what you like better W...

✓ Columna 'text_clean' creada (545,817 filas procesadas)


## 9. Split Estratificado (80/20)

División train/test manteniendo proporciones de las 5 clases.

In [13]:
print("=" * 60)
print("SPLIT ESTRATIFICADO (80/20)")
print("=" * 60)

# Split estratificado por 'emotion'
train_df, test_df = train_test_split(
    df_filtered,
    test_size=0.2,
    stratify=df_filtered['emotion'],
    random_state=42
)

print(f"Train set: {len(train_df):,} filas")
print(f"Test set: {len(test_df):,} filas")
print()

# Verificar proporciones
valid_emotions = ['joy', 'sadness', 'anger', 'fear', 'love']
print("Verificación de proporciones por clase:")
print()
print(f"{'Emoción':<15} {'Train %':<12} {'Test %':<12} {'Diferencia':<12}")
print("=" * 55)

train_dist = train_df['emotion'].value_counts(normalize=True) * 100
test_dist = test_df['emotion'].value_counts(normalize=True) * 100

for emotion in valid_emotions:
    train_pct = train_dist.get(emotion, 0)
    test_pct = test_dist.get(emotion, 0)
    diff = abs(train_pct - test_pct)
    print(f"{emotion:<15} {train_pct:>10.2f}% {test_pct:>10.2f}% {diff:>10.2f}%")

print()
max_diff = max(abs(train_dist.get(e, 0) - test_dist.get(e, 0)) for e in valid_emotions)
if max_diff < 2:
    print(f"✅ Split estratificado exitoso (diferencia máxima: {max_diff:.2f}% < 2%)")
else:
    print(f"⚠️  Diferencia máxima: {max_diff:.2f}% (objetivo: <2%)")

# ── Frequency encoding de Genre (post-split, solo desde train) ───────────────
if GENRE_ENCODING == 'frequency' and 'Genre' in train_df.columns:
    print()
    print("── Frequency encoding de Genre (post-split) ──")
    # Calcular frecuencias SOLO sobre train para evitar data leakage
    genre_freq = train_df['Genre'].value_counts(normalize=True).to_dict()
    train_df = train_df.copy()
    test_df  = test_df.copy()
    train_df['Genre_freq'] = train_df['Genre'].map(genre_freq)
    # Géneros no vistos en train → 0 (frecuencia desconocida)
    test_df['Genre_freq']  = test_df['Genre'].map(genre_freq).fillna(0.0)
    train_df = train_df.drop(columns=['Genre'])
    test_df  = test_df.drop(columns=['Genre'])
    print(f"  ✓ Genre_freq calculada desde train ({len(genre_freq):,} géneros únicos)")
    print(f"  ✓ Géneros nuevos en test → 0.0")
    print(f"  ✓ Columna 'Genre' eliminada de train y test")


SPLIT ESTRATIFICADO (80/20)
Train set: 436,653 filas
Test set: 109,164 filas

Verificación de proporciones por clase:

Emoción         Train %      Test %       Diferencia  
joy                  38.29%      38.29%       0.00%
sadness              31.34%      31.34%       0.00%
anger                20.09%      20.09%       0.00%
fear                  5.15%       5.15%       0.00%
love                  5.12%       5.12%       0.00%

✅ Split estratificado exitoso (diferencia máxima: 0.00% < 2%)

── Frequency encoding de Genre (post-split) ──
  ✓ Genre_freq calculada desde train (3,020 géneros únicos)
  ✓ Géneros nuevos en test → 0.0
  ✓ Columna 'Genre' eliminada de train y test


## 10. Exportación a Parquet

Guardamos train y test en formato parquet para carga rápida.

In [14]:
print("=" * 60)
print("EXPORTACIÓN A PARQUET")
print("=" * 60)

# Exportar train
train_path = '../data/processed/train.parquet'
train_df.to_parquet(train_path, index=False, compression='snappy')
print(f"✓ Train exportado: {train_path}")
print(f"   Tamaño: {len(train_df):,} filas × {len(train_df.columns)} columnas")

# Exportar test
test_path = '../data/processed/test.parquet'
test_df.to_parquet(test_path, index=False, compression='snappy')
print(f"✓ Test exportado: {test_path}")
print(f"   Tamaño: {len(test_df):,} filas × {len(test_df.columns)} columnas")

EXPORTACIÓN A PARQUET
✓ Train exportado: ../data/processed/train.parquet
   Tamaño: 436,653 filas × 28 columnas
✓ Test exportado: ../data/processed/test.parquet
   Tamaño: 109,164 filas × 28 columnas


---

## ✅ RESUMEN DEL NOTEBOOK 02 (Preprocessing)

### Transformaciones Aplicadas

1. **Filtrado de clases**: 500K → ~495K filas (eliminadas 8 clases minoritarias con <5K ejemplos)
2. **Eliminación de columnas**:
   - 9 columnas relacionales (Similar Artist/Song/Score 1/2/3)
   - 4 columnas de metadata (Artist, song, Album, Release Date)
   - **Total eliminadas**: 13 columnas
3. **Manejo de nulos**: <0.1% filas → dropeadas (estrategia más limpia)
4. **Conversiones a numérico** (5 columnas):
   - Length → Length_seconds (227 seg)
   - Loudness (db) → Loudness (-6.85)
   - Time signature → Time_signature (4)
   - Key → Key_encoded (0-23)
   - Explicit → Explicit_binary (0/1)
5. **Encoding categórico**: Genre → one-hot (si <20 géneros) o frequency encoding (Se aplicó Frequency encoding por tener >20 géneros)
6. **Limpieza de texto**: Columna `text_clean` con lyrics normalizadas
7. **Split estratificado**: 80/20 manteniendo proporciones de las 5 clases (±0.1%)

### Artefactos Generados

- `data/processed/train.parquet` (~396K filas, 80%)
- `data/processed/test.parquet` (~99K filas, 20%)

### Columnas Finales

**Numéricas** (~29 columnas):
- Originales: Tempo, Popularity, Energy, Danceability, Positiveness, Speechiness, Liveness, Acousticness, Instrumentalness
- Good for X: 9 columnas binarias (Party, Work/Study, Relaxation, etc.)
- Convertidas: Length_seconds, Loudness, Time_signature, Key_encoded, Explicit_binary

**Categóricas**:
- Genre (Frequency encoding)

**Texto**:
- text, text_clean (para embeddings)

**Target**:
- emotion (5 clases: joy, sadness, anger, fear, love)

### Verificaciones

✅ Proporciones de clases conservadas (diferencia <0.1%)
✅ Reload time <5s por archivo
✅ Sin valores nulos en dataset final
✅ Todas las columnas convertidas a tipos compatibles con modelos

### Próximos Pasos

**Notebook 03**: Generación de embeddings con sentence-transformers (all-MiniLM-L6-v2) + PCA (95% varianza)

---